In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install -Uq wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.8/26.8 MB 63.2 MB/s eta 0:00:00:00:0100:01


In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WANDB_API_KEY")

! wandb login $secret_value_0

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [4]:
import random

import wandb

# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="23f2004513-dl-genai-project",
    # Set the wandb project where this run will be logged.
    project="23f2004513-t22026",
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": 0.02,
        "architecture": "CNN",
        "dataset": "CIFAR-100",
        "epochs": 10,
    },
)

# Simulate training.
epochs = 10
offset = random.random() / 5
for epoch in range(2, epochs):
    acc = 1 - 2**-epoch - random.random() / epoch - offset
    loss = 2**-epoch + random.random() / epoch + offset

    # Log metrics to wandb.
    run.log({"acc": acc, "loss": loss})

# Finish the run and upload any remaining data.
run.finish()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: hanzalatafzeel44 (23f2004513-dl-genai-project) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


acc,▁▅▇▇███▇
loss,█▆▃▁▂▂▁▂
acc,0.89942
loss,0.06523


# Setup

In [5]:
# If a library is missing, uncomment the line below and run it once.
# !pip install pandas numpy scikit-learn gensim nltk sentence-transformers transformers torch -q

import re
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 80)


# 1. Load the Data


In [6]:
DATA_PATH = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nMissing values per column:\n", df.isna().sum())

df.head(3)

Shape: (2000, 8)

Columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']

Missing values per column:
 id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin Heidegger's view on the relati...,Martin Heidegger believes that humans exist within a time continuum that is ...,"Martin Heidegger believes that humans do not exist inside time, but that the...",Martin Heidegger does not believe in the existence of time or that it has an...,Martin Heidegger believes that the relationship between time and human exist...,"Martin Heidegger believes that time is an illusion, and the past, present, a...",B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a technique that uses particle acceler...,Accelerator-based light-ion fusion is a technique that uses particle acceler...,Accelerator-based light-ion fusion is a technique that uses particle acceler...,Accelerator-based light-ion fusion is a technique that uses particle acceler...,Accelerator-based light-ion fusion is a technique that uses particle acceler...,A
2,3,Determine the correct option: What is the term used in astrophysics to descr...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


In [7]:
OPTION_COLS = ["A", "B", "C", "D", "E"]

# Quick sanity check: does every row have a valid answer letter?
print("Unique answer labels:", sorted(df["answer"].unique()))


Unique answer labels: ['A', 'B', 'C', 'D', 'E']


# 2. NLP Foundation & Semantic Similarity

We will build the pipeline step-by-step:

1. **Clean & tokenize text** (and handle missing data)
2. **Generate embeddings** with baseline models: **TF-IDF** and **Word2Vec**
3. **Compute cosine similarity** between the `prompt` and each `option`
4. **Understand & implement mAP@3**
5. Put it together into two simple **baseline models**


### 2.1 Text Cleaning & Tokenization

A single, reusable `clean_text()` function. Keep it simple: lowercase, remove punctuation, collapse extra spaces. We also handle missing / NaN values so nothing breaks later.

In [8]:
def clean_text(text):
    """Lowercase, remove punctuation/numbers-noise, collapse whitespace.
    Returns an empty string for missing values instead of crashing."""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)   # keep only letters/numbers/spaces
    text = re.sub(r"\s+", " ", text).strip()    # collapse multiple spaces
    return text


def tokenize(text):
    """Simple whitespace tokenizer (text is already cleaned)."""
    return text.split()


In [9]:
# Apply cleaning to the prompt and every option column
df["prompt_clean"] = df["prompt"].apply(clean_text)

for col in OPTION_COLS:
    df[col + "_clean"] = df[col].apply(clean_text)

df[["prompt_clean", "A_clean", "B_clean"]].head(3)


,prompt_clean,A_clean,B_clean
0,pick the best possible answer what is martin heidegger s view on the relatio...,martin heidegger believes that humans exist within a time continuum that is ...,martin heidegger believes that humans do not exist inside time but that they...
1,what is accelerator based light ion fusion,accelerator based light ion fusion is a technique that uses particle acceler...,accelerator based light ion fusion is a technique that uses particle acceler...
2,determine the correct option what is the term used in astrophysics to descri...,blueshifting,redshifting


### 2.2 Baseline Embeddings — TF-IDF

TF-IDF turns text into a vector based on word frequency. We fit **one shared vectorizer** on the prompt + all options so every text is represented in the same vector space (required for cosine similarity to make sense).

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer


def build_tfidf_vectorizer(dataframe, option_cols):
    """Fit a single TF-IDF vectorizer on prompts + all options combined."""
    all_texts = list(dataframe["prompt_clean"])
    for col in option_cols:
        all_texts.extend(dataframe[col + "_clean"].tolist())

    vectorizer = TfidfVectorizer()
    vectorizer.fit(all_texts)
    return vectorizer


tfidf_vectorizer = build_tfidf_vectorizer(df, OPTION_COLS)
print("Vocabulary size:", len(tfidf_vectorizer.vocabulary_))


Vocabulary size: 2939


### 2.3 Baseline Embeddings — Word2Vec

Word2Vec learns a dense vector per **word** by predicting neighboring words. We train a small model directly on our own corpus (prompt + options), so no internet download is required — good for explaining the concept quickly.

To turn a *sentence* into one vector, we simply **average the word vectors** in it.


In [11]:
from gensim.models import Word2Vec

WORD2VEC_DIM = 100


def train_word2vec(dataframe, option_cols, vector_size=WORD2VEC_DIM):
    """Train a Word2Vec model on the (cleaned) prompt + option texts."""
    sentences = [tokenize(t) for t in dataframe["prompt_clean"]]
    for col in option_cols:
        sentences += [tokenize(t) for t in dataframe[col + "_clean"]]

    model = Word2Vec(
        sentences=sentences,
        vector_size=vector_size,
        window=5,
        min_count=1,     # keep every word since our corpus is small
        workers=2,
        seed=42,
    )
    return model


def sentence_to_vector(text, w2v_model, dim=WORD2VEC_DIM):
    """Average the Word2Vec vectors of every word in the sentence."""
    words = tokenize(text)
    vectors = [w2v_model.wv[w] for w in words if w in w2v_model.wv]
    if len(vectors) == 0:
        return np.zeros(dim)
    return np.mean(vectors, axis=0)


w2v_model = train_word2vec(df, OPTION_COLS)
print("Word2Vec vocabulary size:", len(w2v_model.wv.index_to_key))


Word2Vec vocabulary size: 2973


### 2.4 Cosine Similarity: Prompt vs Options

Cosine similarity measures how similar two vectors are in *direction* (1 = identical direction, 0 = unrelated, -1 = opposite). We compute it between the prompt vector and each option's vector, then rank the options.

In [12]:
from sklearn.metrics.pairwise import cosine_similarity


def rank_options_tfidf(row, vectorizer, option_cols=OPTION_COLS):
    """Return option letters ranked from most-similar to least-similar (TF-IDF based)."""
    prompt_vec = vectorizer.transform([row["prompt_clean"]])
    scores = {}
    for col in option_cols:
        option_vec = vectorizer.transform([row[col + "_clean"]])
        scores[col] = cosine_similarity(prompt_vec, option_vec)[0][0]
    return sorted(scores, key=scores.get, reverse=True)


def rank_options_word2vec(row, w2v_model, option_cols=OPTION_COLS):
    """Return option letters ranked from most-similar to least-similar (Word2Vec based)."""
    prompt_vec = sentence_to_vector(row["prompt_clean"], w2v_model).reshape(1, -1)
    scores = {}
    for col in option_cols:
        option_vec = sentence_to_vector(row[col + "_clean"], w2v_model).reshape(1, -1)
        scores[col] = cosine_similarity(prompt_vec, option_vec)[0][0]
    return sorted(scores, key=scores.get, reverse=True)


# Quick demo on one row
example = df.iloc[0]
print("Prompt:", example["prompt"][:80], "...")
print("TF-IDF ranking   :", rank_options_tfidf(example, tfidf_vectorizer))
print("Word2Vec ranking :", rank_options_word2vec(example, w2v_model))
print("Correct answer   :", example["answer"])


Prompt: Pick the best possible answer: What is Martin Heidegger's view on the relationsh ...
TF-IDF ranking   : ['C', 'D', 'B', 'A', 'E']
Word2Vec ranking : ['C', 'A', 'D', 'B', 'E']
Correct answer   : B


### 2.5 The Metric — mAP@3

**mAP@3** = Mean Average Precision at 3. For MCQs with a single correct answer, the *Average Precision* for one question is simply:

- `1 / rank` — if the correct answer is within your top-3 predictions (rank 1, 2, or 3)
- `0` — if the correct answer is not in your top-3 at all

Then **mAP@3** is just the average of this score across all questions.

In [13]:
def average_precision_at_3(true_label, predicted_ranking):
    """Average Precision @3 for a single question with one correct answer."""
    top3 = predicted_ranking[:3]
    if true_label in top3:
        rank = top3.index(true_label) + 1   # 1-indexed rank
        return 1.0 / rank
    return 0.0


def mean_average_precision_at_3(true_labels, predicted_rankings):
    """mAP@3 across the whole dataset."""
    scores = [
        average_precision_at_3(t, p)
        for t, p in zip(true_labels, predicted_rankings)
    ]
    return float(np.mean(scores))


# Sanity check with hand-made example
print(average_precision_at_3("B", ["A", "B", "C"]))   # correct at rank 2 -> 0.5
print(average_precision_at_3("B", ["B", "A", "C"]))   # correct at rank 1 -> 1.0
print(average_precision_at_3("B", ["A", "C", "D"]))   # not in top-3   -> 0.0


0.5
1.0
0.0


### 2.6 Baseline Model 1 — TF-IDF Similarity Ranking

Simplest possible baseline: rank the 5 options by TF-IDF cosine similarity to the prompt, take the top 3, and score it with mAP@3.

We use a small sample here for speed — increase `SAMPLE_SIZE` (or use `len(df)`) once you're happy the code runs correctly.

In [14]:
SAMPLE_SIZE = 200   # increase this (or use len(df)) for the real run
sample_df = df.head(SAMPLE_SIZE).copy()

tfidf_predictions = sample_df.apply(
    lambda row: rank_options_tfidf(row, tfidf_vectorizer), axis=1
)

tfidf_score = mean_average_precision_at_3(
    sample_df["answer"].tolist(), tfidf_predictions.tolist()
)
print(f"TF-IDF baseline mAP@3 on {SAMPLE_SIZE} rows: {tfidf_score:.4f}")


TF-IDF baseline mAP@3 on 200 rows: 0.2875


### 2.7 Baseline Model 2 — Word2Vec Similarity Ranking

Exact same idea, but using our Word2Vec sentence vectors instead of TF-IDF.

In [15]:
w2v_predictions = sample_df.apply(
    lambda row: rank_options_word2vec(row, w2v_model), axis=1
)

w2v_score = mean_average_precision_at_3(
    sample_df["answer"].tolist(), w2v_predictions.tolist()
)
print(f"Word2Vec baseline mAP@3 on {SAMPLE_SIZE} rows: {w2v_score:.4f}")


Word2Vec baseline mAP@3 on 200 rows: 0.3267


# 3. Milestone 2 — Enter the Transformers

Here we move from bag-of-words / static-word-vector methods to **context-aware** embeddings
from Transformer models like **BERT** / **RoBERTa**.

> **Note on running this section:** downloading pretrained models (BERT/RoBERTa/Sentence-
> Transformers) requires internet access to Hugging Face. Run this section on **Kaggle**,
> **Google Colab**, or any machine with internet — it will not run inside a fully offline
> sandbox. The code itself is complete and ready to run as-is.


### 3.1 Quick Recap — Why Transformers?

- TF-IDF/Word2Vec give a word or sentence **one fixed vector**, regardless of context. The word *"bank"* gets the same vector in "river bank" and "money bank".
- **BERT/RoBERTa** use the **attention mechanism**: every word's vector is computed *while looking at every other word in the sentence*, so the same word gets a different, context-aware vector depending on the sentence.
- This is exactly what we need for MCQs, where the same option text can be right or wrong depending on how it relates to the specific prompt.

### 3.2 Setup

Install the required libraries (only needed once).

In [16]:
# Run once if these are not already installed:
# !pip install sentence-transformers transformers torch -q

from sentence_transformers import SentenceTransformer


### 3.3 Context-Aware Embeddings with a Pretrained Sentence-Transformer

We use `all-MiniLM-L6-v2` — a small, fast model that is a distilled/fine-tuned version of a BERT-style transformer, specifically trained to produce good **sentence-level** embeddings. This replaces our TF-IDF/Word2Vec functions from Milestone 1 with a single call.

In [17]:
def load_sentence_transformer(model_name="all-MiniLM-L6-v2"):
    """Load a pretrained sentence-transformer (needs internet the first time)."""
    return SentenceTransformer(model_name)


def rank_options_transformer(row, model, option_cols=OPTION_COLS):
    """Rank options by cosine similarity using transformer sentence embeddings."""
    texts = [row["prompt"]] + [row[col] for col in option_cols]
    embeddings = model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)

    prompt_emb = embeddings[0]
    option_embs = embeddings[1:]

    # embeddings are normalized, so dot product == cosine similarity
    scores = {col: float(np.dot(prompt_emb, option_embs[i]))
              for i, col in enumerate(option_cols)}
    return sorted(scores, key=scores.get, reverse=True)


# --- Uncomment to run (needs internet access) ---
st_model = load_sentence_transformer()
example = df.iloc[0]
print("Transformer ranking:", rank_options_transformer(example, st_model))
print("Correct answer     :", example["answer"])


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Transformer ranking: ['C', 'D', 'B', 'E', 'A']
Correct answer     : B


In [19]:
# --- Uncomment to run the full transformer baseline + mAP@3 (needs internet) ---
st_model = load_sentence_transformer()

transformer_predictions = sample_df.apply(
    lambda row: rank_options_transformer(row, st_model), axis=1
)

transformer_score = mean_average_precision_at_3(
    sample_df["answer"].tolist(), transformer_predictions.tolist()
)
print(f"Transformer baseline mAP@3 on {SAMPLE_SIZE} rows: {transformer_score:.4f}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Transformer baseline mAP@3 on 200 rows: 0.4692


### 3.4 Zero-Shot Classification with Transformers

Zero-shot classification lets a model pick the best label for a piece of text **without any task-specific training** — you just give it the text and a list of candidate labels. Here, we treat the `prompt` as the text and the 5 options as the candidate labels, then use the model's confidence scores to rank the options.

In [20]:
from transformers import pipeline


def load_zero_shot_classifier(model_name="facebook/bart-large-mnli"):
    """Load a pretrained zero-shot-classification pipeline (needs internet)."""
    return pipeline("zero-shot-classification", model=model_name)


def rank_options_zero_shot(row, classifier, option_cols=OPTION_COLS):
    """Rank options using zero-shot classification confidence scores."""
    candidate_labels = [row[col] for col in option_cols]
    result = classifier(row["prompt"], candidate_labels)

    # result['labels'] is already sorted by confidence, high to low
    label_to_col = {row[col]: col for col in option_cols}
    ranking = [label_to_col[label] for label in result["labels"]]
    return ranking


# --- Uncomment to run (needs internet + can be slow on CPU) ---
zero_shot_classifier = load_zero_shot_classifier()
example = df.iloc[0]
print("Zero-shot ranking:", rank_options_zero_shot(example, zero_shot_classifier))
print("Correct answer   :", example["answer"])


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Zero-shot ranking: ['B', 'A', 'D', 'C', 'E']
Correct answer   : B


# 4. Comparing All Approaches

Once you've run the transformer cells (on Kaggle/Colab with internet), compare all
approaches side by side using the same `mean_average_precision_at_3` function:


In [21]:
results = {
    "TF-IDF": tfidf_score,
    "Word2Vec": w2v_score,
    # "Sentence-Transformer": transformer_score,   # uncomment once computed above
}

results_df = pd.DataFrame(list(results.items()), columns=["Method", "mAP@3"])
results_df.sort_values("mAP@3", ascending=False)


,Method,mAP@3
1,Word2Vec,0.326667
0,TF-IDF,0.287500


# Baseline Model

In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# -----------------------
# Load Data
# -----------------------

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

# -----------------------
# Convert MCQ -> Binary Classification
# -----------------------

rows = []

for _, row in train.iterrows():

    prompt = str(row["prompt"])

    for option in ["A", "B", "C", "D", "E"]:

        text = prompt + " [SEP] " + str(row[option])

        label = 1 if row["answer"] == option else 0

        rows.append({
            "text": text,
            "label": label
        })

train_binary = pd.DataFrame(rows)

print(train_binary.shape)

# -----------------------
# TF-IDF + Logistic Regression
# -----------------------

model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            max_features=50000,
            ngram_range=(1,2)
        )
    ),
    (
        "clf",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        )
    )
])

model.fit(
    train_binary["text"],
    train_binary["label"]
)

# -----------------------
# Predict Top3
# -----------------------

predictions = []

for _, row in test.iterrows():

    scores = {}

    for option in ["A", "B", "C", "D", "E"]:

        text = str(row["prompt"]) + " [SEP] " + str(row[option])

        prob = model.predict_proba([text])[0][1]

        scores[option] = prob

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    top3 = [x[0] for x in ranked[:3]]

    predictions.append(
        " ".join(top3)
    )

# -----------------------
# Submission
# -----------------------

submission = pd.DataFrame({
    "id": test["id"],
    "Prediction": predictions
})

submission.to_csv(
    "submission.csv",
    index=False
)

submission.head()